In [ ]:
import pandas as pd
from scipy.stats import ks_2samp
import numpy as np

from gsm_benchmarker.results_analyser.prompt_result import MultiPromptResult

from results_notebook_setup import ALPHA, load_results, original_model_order, OUTPUTS_FOLDER, significant_models_path


In [ ]:
N_BOOT = 2000
results_loader = load_results(n_boot=N_BOOT)
full_results = results_loader.full_results

In [ ]:
_ = results_loader.gsm.mres.plot_number_counts(save_prefix=OUTPUTS_FOLDER)

### Evaluating number distribution shift in GSM-Variants w.r.t. GSM-Base

In [ ]:
number_counts = results_loader.gsm.mres.get_number_counts()[0]
numeric_index = pd.to_numeric(number_counts.index)

gsm8k_samples = np.repeat(numeric_index, number_counts['GSM-Base'].values)
main_samples = np.repeat(numeric_index, number_counts['GSM-Variants'].values)

ks_stat, p_value = ks_2samp(gsm8k_samples, main_samples)

print(f"K-S Statistic: {ks_stat:.4f}")
print(f"P-value: {p_value:.4e}")

## Question 1
Are the accuracy drops reported in the GSM-Symbolic paper actually significant?

Evaluating significance of accuracy change on 'main' variant vs 'GSM8K' variant with GSM-Symbolic prompt.

In [ ]:
results_loader.gsm.plot_glmm1()

In [ ]:
results_loader.gsm.glmm1_results

In [ ]:
results_loader.gsm.glmm1_results_to_latex(model_order=original_model_order)

In [ ]:
results_loader.gsm.mres.bootstrap_glmm1

In [ ]:
significant_models = results_loader.gsm.get_significant_models(alpha=ALPHA, drop_only=True)
significant_models

In [ ]:
len(significant_models)

In [ ]:
for key, res in full_results.items():
    if key != 'GSM':
        res.models = significant_models

with open(significant_models_path, 'w') as f:
    f.writelines([f"{m}\n" for m in significant_models])


## Question 2
Do alternative prompt formats remove the variant dependency?

In [ ]:
results_loader.nonformal.plot_glmm1(model_order=significant_models[::-1])

In [ ]:
results_loader.nonformal.glmm1_results_to_latex(model_order=significant_models)

In [ ]:
results_loader.formal.plot_glmm1(model_order=significant_models[::-1])

In [ ]:
results_loader.formal.glmm1_results_to_latex(model_order=significant_models)

In [ ]:
results_loader.short_code.plot_glmm1(model_order=significant_models[::-1])

In [ ]:
results_loader.short_code.glmm1_results_to_latex(model_order=significant_models)

In [ ]:
results_loader.long_code.plot_glmm1(model_order=significant_models[::-1])

In [ ]:
results_loader.long_code.glmm1_results_to_latex(model_order=significant_models)

### Summary of all prompts and models


In [ ]:
all_prompts_result = MultiPromptResult(full_results, save_prefix=OUTPUTS_FOLDER)
all_prompts_result.summary

In [ ]:
fig = all_prompts_result.plot_prompt_comparison(models=significant_models, add_bar_labels=True, x_labels_rotation=5, figsize=(12, 10))

In [ ]:
fig = all_prompts_result.plot_prompt_acc_evolution(models=significant_models, n_cols=2, sharex=False, sharey=False, equal_aspect=False, figsize=(10, 15), bottom_margin=.06)


### Number effect tables

In [ ]:
# number effect - GLMM results
all_prompts_result.glmm2_to_latex("number_effect", models=significant_models)

In [ ]:
# number-effect-corrected variant effect - GLMM results
all_prompts_result.glmm2_to_latex("nec_variant_effect", models=significant_models)